7. Design a governance model for Cyntexa: which columns across which tables are sensitive (PII), which
Unity Catalog groups should have access, and how you'd audit access after the fact.

-->>
For Cyntexa, I would use Unity Catalog to control access to sensitive data.
The main idea is to identify PII columns, mask sensitive information where
required, give access based on team roles, and maintain audit logs.

---

## 1. Identify PII / Sensitive Data

I would classify the important tables and columns like this:

| Table | Sensitive / PII Columns | Data Type |
|---|---|---|
| `customers` | `name`, `email`, `phone`, `address` | PII |
| `customers` | `date_of_birth` | PII / Sensitive |
| `orders` | `customer_id` | Indirect PII |
| `orders` | `order_amount` | Business Sensitive |
| `employees` | `name`, `email`, `phone` | PII |
| `employees` | `address`, `date_of_birth` | PII |
| `employees` | `salary` | Highly Sensitive |
| `payments` | `customer_id` | Indirect PII |
| `payments` | `card_number` | Highly Sensitive |
| `payments` | `bank_account` | Highly Sensitive |
| `products` | `product_id`, `product_name` | Non-PII |
| `products` | `price` | Business Data |

---

## 2. Which Columns Should Be Masked?

I would mask columns where users may need the data for analysis but
do not actually need to see the original value.

### `customers` table

- `email` → partially mask
- `phone` → partially mask
- `address` → hide or restrict
- `date_of_birth` → mask or restrict

Example:

Original:
`rahul.sharma@gmail.com`

Masked:
`r******@gmail.com`

---

### `employees` table

I would be more strict because employee information is sensitive.

- `email` → mask
- `phone` → mask
- `address` → restrict
- `date_of_birth` → restrict
- `salary` → only HR and authorized management

For example:

`salary = 85000`

An analyst should not see the actual salary.

---

### `payments` table

This table needs the highest level of protection.

- `card_number` → always mask
- `bank_account` → always mask
- `customer_id` → restrict based on role

For example:

Original card number:
`4532123456789012`

Displayed value:
`************9012`

Only authorized users should be able to access the original value.

---

%md

## 3. Unity Catalog Groups and Access

I would create separate Unity Catalog groups based on the responsibilities
of each team. The idea is to give users only the access they actually need.

### 1. `data_engineers`

**Purpose:** Build and maintain data pipelines and tables.

**Access:**
- `customers` → Read/Write
- `orders` → Read/Write
- `products` → Read/Write
- `employees` → Limited access
- `payments` → Limited access

**Example:**

A data engineer can read and transform customer data because it may be
required for an ETL pipeline. However, access to highly sensitive
information such as employee salary or complete card numbers should not
be given unless it is required for the job.

---

### 2. `data_analysts`

**Purpose:** Perform reporting, dashboards, and business analysis.

**Access:**
- `customers` → Read access, PII columns masked
- `orders` → Read access
- `products` → Read access
- `employees` → No access to salary and restricted PII
- `payments` → Only aggregated data

**Example:**

An analyst can see:

`customer_id = 1001`

`city = Delhi`

`order_amount = 5000`

But sensitive information such as:

`email = r******@gmail.com`

`phone = ******3210`

should be masked.

The analyst does not need access to complete customer contact details
for normal business reporting.

---

### 3. `data_scientists`

**Purpose:** Build ML models and perform advanced analytics.

**Access:**
- `customers` → Read access, PII masked
- `orders` → Read access
- `products` → Read access
- `employees` → Only required columns
- `payments` → Masked or approved data only

**Example:**

A data scientist may need customer age or order history for a model,
but normally does not need the customer's actual email, phone number,
or card number.

If a sensitive column is genuinely required for a project, access should
be approved before providing it.

---

### 4. `hr_team`

**Purpose:** Manage employee-related information.

**Access:**
- `employees` → Full access to required HR data
- `customers` → No access
- `orders` → No access
- `payments` → No access

**Example:**

HR can access:

`employee_name`

`employee_email`

`employee_phone`

`salary`

because these are required for HR activities.

However, HR does not need access to customer orders or payment card data.

---

### 5. `admins`

**Purpose:** Manage Unity Catalog, permissions, and governance.

**Access:**
- Unity Catalog → Manage
- Catalogs/Schemas → Manage
- Tables → Manage
- Permissions → Manage
- Audit Logs → Read

**Example:**

An admin can grant or revoke permissions for different groups and can
check audit information to see who accessed sensitive data.

Admin access should also be monitored because administrators have
high-level privileges.

---

## 4. Simple Access Summary

| Group | Customers | Orders | Employees | Payments | Products |
|---|---|---|---|---|---|
| `data_engineers` | Read/Write | Read/Write | Limited | Limited | Read/Write |
| `data_analysts` | Read + Masked PII | Read | Restricted | Aggregated | Read |
| `data_scientists` | Read + Masked PII | Read | Required columns only | Masked/Approved | Read |
| `hr_team` | No Access | No Access | Required HR Data | No Access | No Access |
| `admins` | Manage | Manage | Manage | Manage | Manage |

The exact permissions can be changed based on business requirements,
but the main principle is **least privilege**: each team should only
get the minimum access required to perform its work.

8. Extend the SCD Type 2 pattern to track changes across 3+ columns simultaneously, and handle the
edge case of a customer record that hasn't changed since the last load (it should not create a false
new version).

-->>
%md

### SCD Type 2 – Products Table

For the Products table, we can use SCD Type 2 to maintain the history of product changes.

For example, we can track these columns:

- `product_name`
- `category`
- `price`
- `stock_quantity`

### How it works

1. If a new product comes in, insert it as a new record.
2. If any tracked column changes, expire the old record and create **one new version**.
3. If multiple columns change at the same time, create only **one new version** with all the latest values.
4. If nothing has changed, do nothing. This prevents creating duplicate or false versions.

### Example 1 – No Change

Old record:

`P101 | Laptop | Electronics | 50000 | 20`

New record:

`P101 | Laptop | Electronics | 50000 | 20`

Since all tracked columns are the same, we should **not create a new version**.

### Example 2 – One Column Changed

Old record:

`P101 | Laptop | Electronics | 50000 | 20`

New record:

`P101 | Laptop | Electronics | 55000 | 20`

Only the price changed.

So:

- Old record → `is_current = false`
- New record → `is_current = true`
- New version contains price = `55000`

### Example 3 – Multiple Columns Changed

Old record:

`P101 | Laptop | Electronics | 50000 | 20`

New record:

`P101 | Gaming Laptop | Computers | 60000 | 15`

Here, `product_name`, `category`, `price`, and `stock_quantity` changed together.

We should create **only one new SCD Type 2 version**, not one version for each changed column.

### Main Logic

**New product → Insert**

**Any tracked column changed → Expire old record + Insert one new version**

**No tracked column changed → Do nothing**

This keeps the complete product history while avoiding unnecessary duplicate versions.

In [0]:
from pyspark.sql.functions import *

# 1. Source Path (Jahan tumhari Raw CSV files aayengi)
raw_source_path = "/Volumes/cyntexa_dev/day_8/my_volume/customers/"

# 2. Schema Checkpointing Path (Auto Loader state track karne ke liye)
checkpoint_path = "/Volumes/cyntexa_dev/day_8/my_volume/bronze_customers/"
schema_path = "/Volumes/cyntexa_dev/day_8/my_volume/schema_location/"
# 3. Auto Loader Read Stream (Zero Cleaning - Pure Raw Data Capture)
raw_stream_df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation",schema_path) \
    .option("header", "true") \
    .load(raw_source_path) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file_name", col("_metadata.file_name")) \
    .withColumn("source_file_path", col("_metadata.file_path"))

# 4. Write Stream to Bronze Table (Trigger Once taaki ek batch me run ho kar rukk jaaye)
query = raw_stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(availableNow=True) \
    .toTable("cyntexa_dev.day_8.bronze_customers")

#Here the silver table should be created to perform the merge into 

from pyspark.sql.functions import col, trim, lower, row_number, to_timestamp
from pyspark.sql.window import Window

# Bronze read
bronze_df = spark.read.table("cyntexa_dev.day_8.bronze_customers")

# Basic Cleaning (Null Check + String Trimming)
cleaned_df = bronze_df \
    .withColumn("customer_id", trim(col("customer_id"))) \
    .withColumn("first_name", trim(col("first_name"))) \
    .withColumn("last_name", trim(col("last_name"))) \
    .withColumn("email", lower(trim(col("email")))) \
    .withColumn("city", trim(col("city"))) \
    .withColumn("signup_date", trim(col("signup_date"))) \
    .withColumn("updated_at", to_timestamp(col("updated_at"))) \
    .filter(col("customer_id").isNotNull() & (col("customer_id") != "NULL") & (col("customer_id") != "")) \
    .filter(col("updated_at").isNotNull())

# Batch Level Deduplication (Latest updated_at standard pick karega)
window_spec = Window.partitionBy("customer_id").orderBy(col("updated_at").desc())

dedup_cleaned_df = cleaned_df \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

# Temp View for MERGE
dedup_cleaned_df.createOrReplaceTempView("dedup_cleaned_df_customers")

In [0]:
%sql
--  Step 1 ----------------------
merge into  cyntexa_dev.day_8.silver_customers_scd2  t
using dedup_cleaned_df_customers source
on t.customer_id = source.customer_id  and t.end_date is null
when matched and (
    t.email <> source.email or 
    t.city <> source.city or
    t.first_name <> source.first_name or 
    t.last_name <> source.last_name
     )  then update 
     set 
t.end_date = source.updated_at,
t.is_current = false

WHEN NOT MATCHED THEN
  INSERT (
    customer_id, 
    first_name, 
    last_name, 
    email, 
    city, 
    signup_date, 
    start_date, 
    end_date, 
    is_current
  )
  VALUES (
    source.customer_id, 
    source.first_name, 
    source.last_name, 
    source.email, 
    source.city, 
    source.signup_date, 
    source.updated_at, 
    NULL, 
    true
  );

-- step 2---------------
INSERT INTO cyntexa_dev.day_8.silver_customers_scd2  (
  customer_id, first_name, last_name, email, city, signup_date, start_date, end_date, is_current
)
select s.customer_id , s.first_name, s.last_name, s.email, s.city, s.signup_date , s.updated_at, null, true from dedup_cleaned_df_customers  s   
left join  cyntexa_dev.day_8.silver_customers_scd2 t
on s.customer_id = t.customer_id and t.end_date is null  
where t.customer_id is null   

9. (Data Analyst) Using the SCD Type 2 history table, build a customer retention/churn-over-time report
that depends on point-in-time correctness, and explain why a simple 'current state' table would give
the wrong answer here.

-->>
%md

# Customer Retention / Churn Over Time

Using the SCD Type 2 customer history table, we can build a retention and churn report that is **point-in-time correct**.

For each reporting month, we check which SCD2 record was active during that month using:

- `effective_start_date <= report_month`
- `report_month < effective_end_date`

This allows us to know the customer's actual status at that specific point in time.

```sql
WITH report_dates AS (
    SELECT explode(
        sequence(
            to_date('2026-01-01'),
            to_date('2026-12-01'),
            interval 1 month
        )
    ) AS report_month
),

customer_status AS (
    SELECT
        r.report_month,
        c.customer_id,
        c.status
    FROM report_dates r
    JOIN customer_scd2 c
        ON c.effective_start_date <= r.report_month
       AND r.report_month < c.effective_end_date
)

SELECT
    report_month,
    COUNT(DISTINCT CASE
        WHEN status = 'Active' THEN customer_id
    END) AS active_customers,

    COUNT(DISTINCT CASE
        WHEN status = 'Churned' THEN customer_id
    END) AS churned_customers

FROM customer_status
GROUP BY report_month
ORDER BY report_month;
```


## Why SCD Type 2 is Needed for Retention/Churn

SCD Type 2 keeps the complete history of customer status changes.

For example:

`Active → Churned → Active`

A current-state table only stores the latest status, so it would show the customer as `Active` and we would lose the information that the customer was previously `Churned`.

Because retention and churn reports need to know the customer's status **at a specific point in time**, using only the current-state table can give incorrect historical results.

SCD Type 2 solves this by keeping `effective_start_date` and `effective_end_date`, allowing us to identify the customer's correct status for any historical date.

**In short:**  
Current-state table → What is the customer's status now?  
SCD Type 2 → What was the customer's status at that point in time?